In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
import gc
import warnings
warnings.filterwarnings("ignore")

In [ ]:
train = pd.read_parquet("/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/train.parquet")
test = pd.read_parquet("/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/test.parquet")

In [3]:
target = "TARGET"
features = [c for c in train.columns if c not in ["ID","TARGET","CV_GROUP"]]

In [4]:
corr = train[features].corrwith(train["TARGET"]).abs().sort_values(ascending=False)
top_features = corr.head(120).index.tolist()

In [5]:
momentum_train = {}
momentum_test = {}

for f in top_features:

    if "LagT1" in f:

        base = f.replace("_LagT1","")

        lag2 = base + "_LagT2"
        lag3 = base + "_LagT3"

        if lag2 in train.columns and lag3 in train.columns:

            new1 = base + "_momentum"
            new2 = base + "_acceleration"

            momentum_train[new1] = train[f] - train[lag2]
            momentum_test[new1] = test[f] - test[lag2]

            momentum_train[new2] = train[f] - 2*train[lag2] + train[lag3]
            momentum_test[new2] = test[f] - 2*test[lag2] + test[lag3]

In [6]:
lag_features = [f for f in top_features if "Lag" in f]

top_lags = lag_features[:20]

interaction_train = {}
interaction_test = {}

for i in range(len(top_lags)):
    for j in range(i+1, len(top_lags)):

        f1 = top_lags[i]
        f2 = top_lags[j]

        name = "intra_" + f1 + "_x_" + f2

        interaction_train[name] = train[f1] * train[f2]
        interaction_test[name]  = test[f1] * test[f2]

In [7]:
s01_lags = [f for f in top_features if f.startswith("S01") and "Lag" in f]
s02_lags = [f for f in top_features if f.startswith("S02") and "Lag" in f]
s03_lags = [f for f in top_features if f.startswith("S03") and "Lag" in f]

s01_lags = s01_lags[:5]
s02_lags = s02_lags[:5]
s03_lags = s03_lags[:5]

cross_train = {}
cross_test = {}

for f1 in s01_lags:
    for f2 in s02_lags:

        name = "cross12_" + f1 + "_x_" + f2

        cross_train[name] = train[f1] * train[f2]
        cross_test[name]  = test[f1] * test[f2]

for f1 in s01_lags:
    for f2 in s03_lags:

        name = "cross13_" + f1 + "_x_" + f2

        cross_train[name] = train[f1] * train[f2]
        cross_test[name]  = test[f1] * test[f2]

In [8]:
all_train_feats = {}
all_test_feats = {}

all_train_feats.update(cross_train)
all_train_feats.update(momentum_train)
all_train_feats.update(interaction_train)

all_test_feats.update(cross_test)
all_test_feats.update(momentum_test)
all_test_feats.update(interaction_test)

In [9]:
train = pd.concat([train, pd.DataFrame(all_train_feats, index=train.index)], axis=1)
test = pd.concat([test, pd.DataFrame(all_test_feats, index=test.index)], axis=1)


In [ ]:
features = list(dict.fromkeys(
    top_features
    + list(momentum_train.keys())
    + list(interaction_train.keys())
    + list(cross_train.keys())
))

In [11]:
gkf = GroupKFold(n_splits=5)
groups = train["CV_GROUP"]

In [ ]:
pred_A = np.zeros(len(test))

for fold,(train_idx,val_idx) in enumerate(gkf.split(train,groups=groups)):

    X_train = train.iloc[train_idx][features]
    y_train = train.iloc[train_idx][target]

    X_val = train.iloc[val_idx][features]
    y_val = train.iloc[val_idx][target]

    model = lgb.LGBMRegressor(
        n_estimators=2000,
        learning_rate=0.02,
        num_leaves=128,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=5,
        reg_lambda=5,
        random_state=42
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val,y_val)],
        eval_metric="l2",
        callbacks=[lgb.early_stopping(200)]
    )

    pred_A += model.predict(test[features]) / 5
    gc.collect()

In [ ]:
pred_B = np.zeros(len(test))

for fold,(train_idx,val_idx) in enumerate(gkf.split(train,groups=groups)):

    X_train = train.iloc[train_idx][features]
    y_train = train.iloc[train_idx][target]

    X_val = train.iloc[val_idx][features]
    y_val = train.iloc[val_idx][target]

    model = lgb.LGBMRegressor(
        n_estimators=2000,
        learning_rate=0.02,
        num_leaves=128,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.5,
        reg_lambda=0.5,
        random_state=42
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val,y_val)],
        eval_metric="l2",
        callbacks=[lgb.early_stopping(200)]
    )

    pred_B += model.predict(test[features]) / 5
    gc.collect()

In [14]:
pred_final = 0.72 * pred_A + 0.28 * pred_B

In [15]:
submission = pd.DataFrame({
    "ID": test["ID"],
    "TARGET": pred_final
})

submission.to_csv("submission.csv",index=False)